# TNP conformer sensitivity -- SimpleFold arm, and a seed replication (GPU)

Two jobs in one run, because they need the same folds.

**1. TNP.** SimpleFold's compactness against the crystal, plus the thing no other model can give:
the **distribution** of compactness over 6 independent folds of one sequence. How many nanobodies
have a compactness spread that straddles the kinked/extended antimode at 0.961?

**2. Replication.** The spread-predicts-error result currently rests on n=48 at **one** seed --
a re-run with fixed batch composition returned byte-identical numbers, so it was never replicated.
This run uses **seed 777** and **all 597 targets**, which is a 12x larger, genuinely independent test.

Targets are processed in three chunks. Each chunk is scored and written before the next starts, so
a wall-clock timeout still leaves completed chunks on disk.


In [ ]:
import os, sys, json, time, glob, subprocess, shutil
import numpy as np
NS, STEPS, TAU, SEED, MODEL = '6','500','0.01','777','simplefold_100M'
NCHUNK = 3
WORK, SCRATCH = '/kaggle/working','/tmp/sf'
T0=time.time(); os.makedirs(SCRATCH,exist_ok=True)
R={'note':'TNP compactness distribution + spread-vs-error replication at a NEW seed',
   'model':MODEL,'n_samples':int(NS),'tau':float(TAU),'seed':int(SEED),'rows':[]}
def note(t):
    R['stage']=t; R['elapsed_min']=round((time.time()-T0)/60,1)
    json.dump(R,open(f'{WORK}/tnp_sf.json','w'),indent=1,default=str)
    print(t,R['elapsed_min'],'min',flush=True)

import torch
n=torch.cuda.device_count(); cap=torch.cuda.get_device_capability(0)
R['gpus']=[torch.cuda.get_device_name(g) for g in range(n)]
assert n==2, f'need T4 x2, got {n}: {R["gpus"]}'
assert cap>=(7,0), f'{R["gpus"][0]} is sm_{cap[0]}{cap[1]}; needs sm_70+'
_ds=glob.glob('/kaggle/input/**/targets_vhh.json',recursive=True); assert len(_ds)==1,_ds
TG={f"{t['pdb']}{t['H']}":t for t in json.load(open(_ds[0]))}
print(R['gpus'],len(TG),'targets',flush=True)

def must(c,w):
    p=subprocess.run(c,shell=True,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(f'{w} (rc={p.returncode}):\n'+(p.stdout+p.stderr)[-2000:])
must('git clone -q --depth 1 -b fix-multichain-fasta-parsing '
     'https://github.com/omar-A-hassan/ml-simplefold '+SCRATCH+'/repo','clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo','install simplefold')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git','install esm')
must(f'{sys.executable} -m pip install -q biotite','install biotite')
src=open(f'{SCRATCH}/repo/src/simplefold/utils/fasta_utils.py').read()
assert 'chr(ord("A") + i)' in src,'multichain fix missing -- wrong branch?'
INF=f'{SCRATCH}/repo/src/simplefold/inference.py'; s=open(INF).read()
a='        esm_model = esm_model.to(device)'
assert s.count(a)==1, f'fp16 anchor found {s.count(a)}x'
open(INF,'w').write(s.replace(a,'        esm_model = esm_model.half().to(device)'))
print('fp16 patched',flush=True)
note('installed')


In [ ]:

import numpy as np

def _ang(a,b,c):
    u,v=a-b,c-b
    return float(np.degrees(np.arccos(np.clip(np.dot(u,v)/(np.linalg.norm(u)*np.linalg.norm(v)),-1,1))))

def _dih(a,b,c,d):
    b0,b1,b2=a-b,c-b,d-c; b1=b1/np.linalg.norm(b1)
    v=b0-np.dot(b0,b1)*b1; w=b2-np.dot(b2,b1)*b1
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1,v),w),np.dot(v,w))))

def compactness(ca, span):
    """TNP: n residues in CDR3 / reach; reach = anchors (IMGT 105,117) to furthest loop point."""
    lo,hi=span; loop=ca[lo:hi]; ok=np.isfinite(loop[:,0]); n=int(hi-lo)
    if ok.sum()<3 or not (np.isfinite(ca[lo,0]) and np.isfinite(ca[hi-1,0])): return None,None,n
    base=(ca[lo]+ca[hi-1])/2.0
    reach=float(np.linalg.norm(loop[ok]-base,axis=-1).max())
    return (n/reach if reach>0 else None), reach, n

def kink(ca, span):
    hi=span[1]; idx=[hi-3,hi-2,hi-1,hi]        # IMGT 115,116,117,118
    if idx[0]<0 or idx[3]>=len(ca): return None,None,'na'
    p=ca[idx]
    if not np.isfinite(p[:,0]).all(): return None,None,'na'
    tau=_ang(p[0],p[1],p[2]); alpha=_dih(p[0],p[1],p[2],p[3])
    if 0<alpha<120 and 85<tau<130: lab='kinked'
    elif alpha<-100 and 100<tau<145: lab='extended'
    else: lab='other'
    return alpha,tau,lab

# boundaries derived from the 597 crystals in this dataset (TNP Table 2 is an image)
ANTIMODE, P5, P95 = 0.961, 0.736, 1.294
def band(v):
    if v is None: return 'na'
    return 'amber_low' if v<P5 else 'amber_high' if v>P95 else 'green'
def conf(v):
    return 'na' if v is None else ('kinked' if v>=ANTIMODE else 'extended')

def ca_from_hca(hca):
    return np.array([[np.nan]*3 if r is None or r[1] is None else r[1] for r in hca],float)

def kabsch(P,Q):
    Pm,Qm=P.mean(0),Q.mean(0)
    V,_,W=np.linalg.svd((P-Pm).T@(Q-Qm)); D=np.diag([1,1,np.sign(np.linalg.det(V@W))])
    return V@D@W,Pm,Qm

def cdr3_rmsd(ca, ref, span):
    """framework-aligned CDR3 CA RMSD"""
    L=len(ref); c3=np.zeros(L,bool); c3[span[0]:span[1]]=True
    ok=np.isfinite(ref[:,0])&np.isfinite(ca[:,0])
    fwsel=ok&~c3
    if fwsel.sum()<40: return None
    R,Pm,Qm=kabsch(ca[fwsel],ref[fwsel])
    al=(ca-Pm)@R+Qm
    s=ok&c3
    if s.sum()<3: return None
    return float(np.sqrt(((al[s]-ref[s])**2).sum(-1).mean()))

import biotite.structure.io.pdbx as pdbx
def ca_from_cif(fp,L):
    st=pdbx.get_structure(pdbx.CIFFile.read(fp),model=1)
    u=np.unique(st.res_id)
    if len(np.unique(st.chain_id))!=1 or len(u)!=L or u.max()-u.min()+1!=L: return None
    P=np.full((L,3),np.nan); m=st.atom_name=='CA'
    P[st.res_id[m]-st.res_id.min()]=st.coord[m]
    return P
print('TNP + parse helpers loaded',flush=True)


In [ ]:
fadir=f'{SCRATCH}/fa'; os.makedirs(fadir,exist_ok=True)
for tid,t in TG.items(): open(f'{fadir}/{tid}.fasta','w').write('>A|'+tid+'\n'+t['hseq']+'\n')
w=subprocess.run([sys.executable,'-c',
  "import torch,esm; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],
  capture_output=True,text=True)
if w.returncode:
    raise RuntimeError('esm warm-up failed; aborting rather than racing:\n'+(w.stdout+w.stderr)[-2000:])
print('esm cache warm',flush=True); note('warmed')

tids=sorted(TG); chunks=[tids[i::NCHUNK] for i in range(NCHUNK)]
for ci,chunk in enumerate(chunks):
    tag=f'c{ci}'
    for g in (0,1):
        d=f'{SCRATCH}/{tag}_sh{g}'; shutil.rmtree(d,ignore_errors=True); os.makedirs(d)
    for i,tid in enumerate(chunk): os.link(f'{fadir}/{tid}.fasta',f'{SCRATCH}/{tag}_sh{i%2}/{tid}.fasta')
    t0=time.time(); procs=[]
    for g in (0,1):
        if g: time.sleep(60)
        procs.append(subprocess.Popen(
            ['simplefold','--simplefold_model',MODEL,'--backend','torch',
             '--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,'--tau',TAU,
             '--nsample_per_protein',NS,'--seed',SEED,
             '--fasta_path',f'{SCRATCH}/{tag}_sh{g}','--output_dir',f'{SCRATCH}/{tag}_out{g}'],
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
            env={**os.environ,'CUDA_VISIBLE_DEVICES':str(g)}))
        print(f' chunk{ci} launched gpu{g}',flush=True)
    rc={}
    for g,p in enumerate(procs):
        o=p.communicate()[0]; rc[f'gpu{g}']=p.returncode
        if p.returncode: print(f' chunk{ci} GPU{g} FAILED\n',o[-2000:],flush=True)
    R.setdefault('chunk_rc',{})[ci]=rc
    R.setdefault('chunk_min',{})[ci]=round((time.time()-t0)/60,1)

    for tid in chunk:
        t=TG[tid]; L=len(t['hseq']); span=tuple(t['cdr3']); ref=ca_from_hca(t['hca'])
        files=sorted(glob.glob(f'{SCRATCH}/{tag}_out*/**/*{tid}*.cif',recursive=True))
        S=[ca_from_cif(f,L) for f in files]; S=[x for x in S if x is not None]
        if len(S)<2: R.setdefault('skip',[]).append(tid); continue
        cc,cr,nres=compactness(ref,span)
        comp=[compactness(x,span)[0] for x in S]; comp=[c for c in comp if c is not None]
        err=[cdr3_rmsd(x,ref,span) for x in S]; err=[e for e in err if e is not None]
        if not comp or not err: R.setdefault('skip',[]).append(tid+':score'); continue
        base=S[0]; AL=[base]
        for P in S[1:]:
            m=np.isfinite(P[:,0])&np.isfinite(base[:,0])
            m[span[0]:span[1]]=False
            Rot,Pm,Qm=kabsch(P[m],base[m]); AL.append((P-Pm)@Rot+Qm)
        AL=np.stack(AL)
        spread=float(np.nanmean(np.nanstd(AL[:,span[0]:span[1]],axis=0)))
        confs={conf(c) for c in comp}
        R['rows'].append({'tid':tid,'n_res':nres,'cdr3_len':span[1]-span[0],
            'cry_compact':cc,'cry_conf':conf(cc),'cry_band':band(cc),'cry_kink':kink(ref,span)[2],
            'sf_compact':round(float(np.median(comp)),4),'sf_conf':conf(float(np.median(comp))),
            'sf_band':band(float(np.median(comp))),
            'compact_min':round(min(comp),4),'compact_max':round(max(comp),4),
            'straddles':len(confs)>1,
            'err':round(err[0],3),'err_med':round(float(np.median(err)),3),
            'err_best':round(min(err),3),'spread_cdr3':round(spread,4),'n':len(S)})
    note(f'chunk{ci}')
    for g in (0,1): shutil.rmtree(f'{SCRATCH}/{tag}_out{g}',ignore_errors=True)
    print(f' chunk{ci} done, {len(R["rows"])} rows total',flush=True)
note('folded')


In [ ]:
rows=R['rows']
def rank(v):
    o=np.argsort(np.argsort(np.asarray(v,float))); return o.astype(float)+1
def sp(a,b):
    ra,rb=rank(a),rank(b); return float(np.corrcoef(ra,rb)[0,1])
def partial(a,b,c):
    r=lambda x,y: sp(x,y)
    rab,rac,rbc=r(a,b),r(a,c),r(b,c)
    return (rab-rac*rbc)/np.sqrt((1-rac**2)*(1-rbc**2))

ok=[r for r in rows if r['cry_compact'] is not None]
print('=== 1. REPLICATION: spread vs error, seed 777, n=%d ==='%len(rows))
sprd=[r['spread_cdr3'] for r in rows]; err=[r['err'] for r in rows]; ln=[r['cdr3_len'] for r in rows]
rho=sp(sprd,err); rho_p=partial(sprd,err,ln)
rng=np.random.default_rng(0); idx=np.arange(len(rows))
bs=sorted(sp([sprd[i] for i in s],[err[i] for i in s]) for s in (rng.choice(idx,len(idx)) for _ in range(4000)))
R['replication']={'n':len(rows),'rho':round(rho,4),'rho_ctrl_len':round(rho_p,4),
                  'ci95':[round(bs[100],4),round(bs[3900],4)],'frac_le_zero':round(sum(1 for x in bs if x<=0)/4000,5),
                  'prior_claim':{'n':48,'rho':0.572,'seed':12345}}
print(json.dumps(R['replication'],indent=1))

print('\n=== 2. TNP: SimpleFold compactness vs crystal, n=%d ==='%len(ok))
d=[abs(r['sf_compact']-r['cry_compact']) for r in ok]
rel=[abs(r['sf_compact']-r['cry_compact'])/r['cry_compact'] for r in ok]
R['tnp']={'n':len(ok),
  'median_abs_dcompact':round(float(np.median(d)),4),
  'median_rel_err_pct':round(100*float(np.median(rel)),2),
  'p95_rel_err_pct':round(100*float(np.percentile(rel,95)),2),
  'band_flip_pct':round(100*sum(1 for r in ok if r['sf_band']!=r['cry_band'])/len(ok),2),
  'conf_flip_pct':round(100*sum(1 for r in ok if r['sf_conf']!=r['cry_conf'])/len(ok),2),
  'straddle_pct':round(100*sum(1 for r in ok if r['straddles'])/len(ok),2),
  'rho_compact_err':round(sp([abs(r['sf_compact']-r['cry_compact']) for r in ok],
                             [r['err'] for r in ok]),4)}
print(json.dumps(R['tnp'],indent=1))
print('\nstraddle_pct = nanobodies whose 6 folds do not agree on kinked vs extended.')
print('That number is the thing only a generative model can report.')
note('done')
